# 10 — Baselines multi-horizonte  (auto-generado por build_notebook_10.py)

Evalúa baselines clásicos (B0 media global, B1 persistencia naive,
B2 media móvil w∈{5,10,15}, B3 suavizado exponencial simple α=0.3,
B4 promedio histórico por hora del día) **y un baseline ajustado**
(B5_XGB, gradient boosting sobre 12 lags + features de calendario/slot)
sobre los corredores **E2** y **E59** a horizontes **h ∈ {1, 3, 5, 10} minutos**.

B1/B2/B3/B5_XGB son horizon-aware. B0/B4 son horizon-agnósticos: su predicción
no depende del horizonte. El CSV de salida incluye columna `horizon`.

In [ ]:

import hashlib

import polars as pl
import numpy as np
from pathlib import Path

# Frozen SHA-256 of the hash-gated inputs (same contract as the DL notebooks).
# atypical_days.csv is a REQUIRED input for B5_XGB: the run stops before
# fitting when it is missing or its bytes differ from the pinned snapshot.
INPUT_HASHES = {
    "atypical_days.csv": "2054245cc830e58b9397b75ea3b55d034581046b64e73b1630ca7d464e3ecb86",
}

def _sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

# Locate a required input by filename and verify its frozen SHA-256.
def _resolve_input(name: str) -> Path:
    roots = [Path("/kaggle/input"), Path(".")]
    candidates = [p for root in roots if root.exists() for p in sorted(root.rglob(name))]
    if not candidates:
        raise FileNotFoundError(f"Required input not found anywhere: {name}")
    for path in candidates:
        if _sha256_file(path) == INPUT_HASHES[name]:
            return path
    raise ValueError(
        f"No copy of {name} matches its frozen SHA-256 — "
        f"candidates: {[str(p) for p in candidates]}"
    )

# Locate headways parquets for E2 and E59 under /kaggle/input or local dir.
def _find_parquet(empresa_id: int) -> Path:
    name = f"headways_E{empresa_id}.parquet"
    if Path("/kaggle/input").exists():
        candidates = list(Path("/kaggle/input").rglob(name))
        if candidates:
            return candidates[0]
    candidates = list(Path(".").rglob(name))
    if candidates:
        return candidates[0]
    raise FileNotFoundError(
        f"{name} not found. Expected at /kaggle/input/**/{name}"
    )

OUTPUT_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path(".")
OUTPUT_DIR.mkdir(exist_ok=True)
CSV_OUT = OUTPUT_DIR / "baselines_results_multih.csv"
RESID_OUT = OUTPUT_DIR / "xgb_residuals_multih.csv"
SEARCH_OUT = OUTPUT_DIR / "xgb_search_config_multih.csv"

print(f"Output dir: {OUTPUT_DIR}")

## Module: evaluation/splits

Temporal split helper (`split_temporal`) and train-only p99 winsorization
(`winsorize_train_p99`).  Split date ranges are locked in spec §3.

In [ ]:
"""Temporal split and winsorization helpers for headway evaluation — Fase 3.

Public API:
    split_temporal(df: pl.DataFrame) -> pl.DataFrame
    winsorize_train_p99(df: pl.DataFrame) -> tuple[pl.DataFrame, float]

Constants (split date ranges, locked in spec §3 and design §5):
    SPLIT_TRAIN_START, SPLIT_TRAIN_END
    SPLIT_VAL_START,   SPLIT_VAL_END
    SPLIT_TEST_START,  SPLIT_TEST_END
    WINSOR_QUANTILE

Design decisions (locked in design §5 and §9):
  - Split key is pl.col("t").dt.date() membership, NOT row index.
  - Three ranges are exhaustive and mutually exclusive.
  - Rows outside all three ranges receive None (split column = null).
  - Winsorization threshold is computed on train rows only (AC-WINSOR-1, AC-WINSOR-2).
  - Null delta_t_min rows are NOT clipped (AC-WINSOR-3).
  - Rows above threshold are clipped (not dropped) (AC-WINSOR-4).
  - Constants live here (not PRODUCTIVE_PARAMS) — evaluation protocol concern.
  - WINSOR_QUANTILE and split dates are not added to pyproject.toml.
"""
from __future__ import annotations

from datetime import date

import polars as pl

# ---------------------------------------------------------------------------
# Split date range constants (spec §3, inclusive on both ends)
# ---------------------------------------------------------------------------

SPLIT_TRAIN_START: date = date(2023, 10, 1)
SPLIT_TRAIN_END:   date = date(2024, 1, 15)

SPLIT_VAL_START:   date = date(2024, 1, 16)
SPLIT_VAL_END:     date = date(2024, 2, 7)

SPLIT_TEST_START:  date = date(2024, 2, 8)
SPLIT_TEST_END:    date = date(2024, 2, 29)

WINSOR_QUANTILE: float = 0.99


def split_temporal(df: pl.DataFrame) -> pl.DataFrame:
    """Add a `split` column (Utf8) with values {"train", "val", "test"}.

    Membership is determined by pl.col("t").dt.date() against the six
    module-level date constants.  Rows outside all three ranges receive
    null (should not exist in the R7 v4 dataset; harness raises if found).

    Parameters
    ----------
    df:
        headways DataFrame containing at least a `t` (Datetime) column.

    Returns
    -------
    pl.DataFrame — input frame with one added column `split: Utf8`.
    """
    day = pl.col("t").dt.date()
    return df.with_columns(
        pl.when((day >= SPLIT_TRAIN_START) & (day <= SPLIT_TRAIN_END))
          .then(pl.lit("train"))
          .when((day >= SPLIT_VAL_START) & (day <= SPLIT_VAL_END))
          .then(pl.lit("val"))
          .when((day >= SPLIT_TEST_START) & (day <= SPLIT_TEST_END))
          .then(pl.lit("test"))
          .otherwise(None)
          .alias("split")
    )


def winsorize_train_p99(
    df: pl.DataFrame,
) -> tuple[pl.DataFrame, float]:
    """Clip delta_t_min to the 99th-percentile threshold computed on train rows only.

    The threshold is computed once as a scalar from non-null train-split rows.
    It is then applied as a clip ceiling to ALL rows (train + val + test).
    Null delta_t_min values are never clipped — they remain null (AC-WINSOR-3).

    Parameters
    ----------
    df:
        headways DataFrame that already has a `split` column (added by
        split_temporal) and a `delta_t_min` (Float64 nullable) column.

    Returns
    -------
    (clipped_df, threshold)
        clipped_df: same schema as df, delta_t_min clipped.
        threshold: the scalar train-p99 value used as the clip ceiling.

    Design note (AC-WINSOR-2 leakage guard):
        The filter `split == "train"` is applied BEFORE computing the quantile,
        so extreme outliers in val or test rows cannot shift the threshold.
    """
    threshold = float(
        df.filter(
            (pl.col("split") == "train") & pl.col("delta_t_min").is_not_null()
        )["delta_t_min"]
        .quantile(WINSOR_QUANTILE)
    )

    # Clip: preserve null rows; clip non-null rows to threshold from above.
    # pl.min_horizontal(col, lit(threshold)) would coerce null → 0 in some
    # polars versions, so we use the explicit when/then pattern (design §5).
    clipped = df.with_columns(
        pl.when(pl.col("delta_t_min").is_null())
          .then(None)
          .otherwise(
              pl.min_horizontal(pl.col("delta_t_min"), pl.lit(threshold))
          )
          .alias("delta_t_min")
    )
    return clipped, threshold

## Module: evaluation/metrics

`mae` and `rmse` in minutes.  Both accept polars Series or numpy arrays.
Null/NaN rows are dropped before aggregation.  MAPE is explicitly excluded
(spec B3-NO-MAPE).

In [ ]:
"""Evaluation metrics for headway forecasting — Fase 3.

Public API:
    mae(y_true, y_pred) -> float
    rmse(y_true, y_pred) -> float

Both functions accept polars Series (Float64) or numpy arrays (float64).
Null / NaN masking: rows where EITHER y_true or y_pred is null/NaN are
dropped before aggregation.  If no valid rows remain, ValueError is raised.

Design decisions locked in design §4:
  - ValueError on empty/all-null input (NOT silent NaN return).
  - Only MAE and RMSE are in scope (spec B3-NO-MAPE — ratio-based metrics
    are out of scope because near-zero headways cause denominator blow-up).
  - No new pyproject.toml dependencies (polars + numpy already present).
"""
from __future__ import annotations

import numpy as np
import polars as pl


def _to_numpy_with_mask(
    y_true: pl.Series | np.ndarray,
    y_pred: pl.Series | np.ndarray,
) -> tuple[np.ndarray, np.ndarray]:
    """Coerce both inputs to float64 numpy arrays and apply the null/NaN mask.

    Polars Series with dtype Float64: null cells become NaN via .to_numpy().
    numpy arrays: assumed to already use NaN for missing values.

    Returns
    -------
    (y_true_masked, y_pred_masked) — two 1-D float64 arrays of equal length,
    containing no NaN values.  May be empty if all rows were masked.
    """
    # Coerce to numpy.
    if isinstance(y_true, pl.Series):
        yt = y_true.to_numpy(allow_copy=True).astype(np.float64)
    else:
        yt = np.asarray(y_true, dtype=np.float64).ravel()

    if isinstance(y_pred, pl.Series):
        yp = y_pred.to_numpy(allow_copy=True).astype(np.float64)
    else:
        yp = np.asarray(y_pred, dtype=np.float64).ravel()

    # Elementwise mask: keep row only if BOTH sides are finite (not NaN).
    mask = ~(np.isnan(yt) | np.isnan(yp))
    return yt[mask], yp[mask]


def mae(
    y_true: pl.Series | np.ndarray,
    y_pred: pl.Series | np.ndarray,
) -> float:
    """Mean Absolute Error in minutes, with null/NaN masking.

    Parameters
    ----------
    y_true, y_pred:
        Ground-truth and predicted headway values in minutes.
        Accepts polars Series (Float64) or numpy arrays (float64).
        Null / NaN positions in either input are dropped before computation.

    Returns
    -------
    float — MAE in minutes.

    Raises
    ------
    ValueError
        If the masked input is empty (all-null or zero-length).
    """
    yt, yp = _to_numpy_with_mask(y_true, y_pred)
    if len(yt) == 0:
        raise ValueError(
            "mae: metric on empty/all-null input — no valid (y_true, y_pred) pairs."
        )
    return float(np.mean(np.abs(yt - yp)))


def rmse(
    y_true: pl.Series | np.ndarray,
    y_pred: pl.Series | np.ndarray,
) -> float:
    """Root Mean Squared Error in minutes, with null/NaN masking.

    Parameters
    ----------
    y_true, y_pred:
        Ground-truth and predicted headway values in minutes.
        Accepts polars Series (Float64) or numpy arrays (float64).
        Null / NaN positions in either input are dropped before computation.

    Returns
    -------
    float — RMSE in minutes.

    Raises
    ------
    ValueError
        If the masked input is empty (all-null or zero-length).
    """
    yt, yp = _to_numpy_with_mask(y_true, y_pred)
    if len(yt) == 0:
        raise ValueError(
            "rmse: metric on empty/all-null input — no valid (y_true, y_pred) pairs."
        )
    return float(np.sqrt(np.mean((yt - yp) ** 2)))

## Module: baselines/statistical

B0 global mean, B1 naive persistence (horizon-aware), B2 moving average
(w∈{5,10,15}, horizon-aware), B3 simple exponential smoothing (α=0.3,
horizon-aware), B4 historical average per hour.
All operate per slot `(empresaid, direction, pair_rank)`.

In [ ]:
"""Classical statistical baselines for headway forecasting — Fase 3.

Public API:
    predict_b0(headways: pl.DataFrame) -> pl.DataFrame
    predict_b1(headways: pl.DataFrame, *, horizon: int = 1) -> pl.DataFrame
    predict_b2(headways: pl.DataFrame, *, window: int, horizon: int = 1) -> pl.DataFrame
    predict_b3(headways: pl.DataFrame, *, alpha: float = SES_ALPHA, horizon: int = 1) -> pl.DataFrame
    predict_b4_ha(headways: pl.DataFrame) -> pl.DataFrame

Input contract (all four functions):
    The DataFrame must have a `split` column (Utf8) added by split_temporal.
    Columns consumed: empresaid, t, direction, pair_rank, delta_t_min, split.

Output contract:
    Each function returns the input frame with ONE additional column:
        B0 → y_pred_b0
        B1 → y_pred_b1
        B2 → y_pred_b2_w{window}
        B3 → y_pred_b3

Predictions are filled for ALL rows (train + test); the evaluation harness
consumes test rows only.  Filling train rows costs negligibly more and lets
future SDDs reuse predictions if needed (design §3).

Design decisions locked in design §3 and §9:
  - Functions, not classes (consistent with project precedent).
  - Slot key: (empresaid, direction, pair_rank).
  - B2 `window` = count of last NON-NULL observations (not a time window).
  - B2 min_periods = window // 2  (floor division).
  - B3 alpha = SES_ALPHA = 0.3, per-slot online recursion, null-skip.
  - B3 state init: NaN until first non-null train obs; first non-null sets s directly.
  - No new pyproject.toml dependencies (polars + numpy only).
"""
from __future__ import annotations

import numpy as np
import polars as pl

# ---------------------------------------------------------------------------
# Module-level constants (locked in design §9)
# ---------------------------------------------------------------------------

BASELINE_B2_WINDOWS: tuple[int, ...] = (5, 10, 15)
SES_ALPHA: float = 0.3

_SLOT_COLS: list[str] = ["empresaid", "direction", "pair_rank"]


# ===========================================================================
# B0 — Global mean per slot (train rows only)
# ===========================================================================

def predict_b0(headways: pl.DataFrame) -> pl.DataFrame:
    """Add column `y_pred_b0`: per-slot mean of train delta_t_min.

    The prediction is constant within a slot — the arithmetic mean of all
    non-null delta_t_min values in the train split for that slot.  Slots
    with no non-null train observations receive null (AC-B0-2).

    Parameters
    ----------
    headways:
        headways DataFrame with `split` column attached.

    Returns
    -------
    pl.DataFrame — input frame with `y_pred_b0` (Float64 nullable) added.
    """
    train_means = (
        headways
        .filter(pl.col("split") == "train")
        .group_by(_SLOT_COLS)
        .agg(pl.col("delta_t_min").mean().alias("y_pred_b0"))
    )
    return headways.join(train_means, on=_SLOT_COLS, how="left")


# ===========================================================================
# B1 — Naive / persistence baseline
# ===========================================================================

def predict_b1(headways: pl.DataFrame, *, horizon: int = 1) -> pl.DataFrame:
    """Add column `y_pred_b1`: last non-null delta_t_min seen `horizon` steps before each row.

    Uses forward_fill().shift(horizon).over(slot) — the canonical polars pattern for
    ŷ_{t+h} = y_t with null gaps.  Causal by construction (shift prevents the
    current-row value from appearing as its own prediction).

    Parameters
    ----------
    headways:
        headways DataFrame with `split` column attached.
    horizon:
        Number of steps to shift. Default 1 reproduces the original behavior.
        Calling ``predict_b1(df)`` (no horizon arg) is identical to ``predict_b1(df, horizon=1)``.

    Returns
    -------
    pl.DataFrame — input frame sorted by (slot, t), with `y_pred_b1` added.
    """
    return (
        headways
        .sort(_SLOT_COLS + ["t"])
        .with_columns(
            pl.col("delta_t_min")
              .forward_fill()
              .shift(horizon)
              .over(_SLOT_COLS)
              .alias("y_pred_b1")
        )
    )


# ===========================================================================
# B2 — Trailing moving average of last w NON-NULL observations
# ===========================================================================

def predict_b2(headways: pl.DataFrame, *, window: int, horizon: int = 1) -> pl.DataFrame:
    """Add column `y_pred_b2_w{window}`: mean of last `window` non-null observations.

    Semantics (locked, design §3):
      - `window` is a COUNT of non-null observations, not a time window.
      - min_periods = window // 2  (floor).
      - Prediction at row i uses only observations strictly before row i (causal).

    Horizon rule (Fase 6.5):
      The 1-step prediction is computed first (rolling_mean.shift(1) on the
      non-null sub-series + join_asof backward).  Then shift(horizon-1) is
      applied over the slot key so that ŷ_{t+h} = ŷ_{t+1} lagged by h-1
      additional steps.  horizon=1 → shift(0) = identity (backward-compatible).

    Implementation:
      - group_by(slot).map_groups(lambda g: _b2_one_slot(g, window))
      - Within each group: extract non-null values, compute rolling_mean with
        shift(1) (causal), then join_asof(strategy="backward") back to
        original group rows on `t`.
      - After concat, apply shift(horizon-1).over(_SLOT_COLS).

    Parameters
    ----------
    headways:
        headways DataFrame with `split` column attached.
    window:
        Number of non-null observations in the trailing window.
    horizon:
        Prediction horizon in steps.  Default 1 reproduces the original
        behavior exactly.

    Returns
    -------
    pl.DataFrame — input frame with `y_pred_b2_w{window}` (Float64 nullable) added.
    """
    col_name = f"y_pred_b2_w{window}"
    min_periods = window // 2

    def _b2_one_slot(group: pl.DataFrame) -> pl.DataFrame:
        group = group.sort("t")

        # Extract non-null rows only, in time order.
        non_null = group.filter(pl.col("delta_t_min").is_not_null())

        if len(non_null) == 0:
            # No non-null observations: all predictions are null.
            return group.with_columns(pl.lit(None, dtype=pl.Float64).alias(col_name))

        # Compute rolling mean on the non-null sub-series, then shift(1) for
        # causality: the prediction at position i uses observations 0..i-1.
        non_null = non_null.with_columns(
            pl.col("delta_t_min")
              .rolling_mean(window_size=window, min_samples=min_periods)
              .shift(1)
              .alias(col_name)
        )

        # Align back to the full group (including null rows) via join_asof.
        # strategy="backward" finds the most recent non-null rolling mean at or
        # before each timestamp in the original group.
        result = group.join_asof(
            non_null.select(["t", col_name]),
            on="t",
            strategy="backward",
        )
        return result

    sorted_df = headways.sort(_SLOT_COLS + ["t"])
    slots = sorted_df.partition_by(_SLOT_COLS, maintain_order=True)
    result = pl.concat([_b2_one_slot(g) for g in slots])

    # Apply horizon shift: shift(horizon-1) over slot so predictions look
    # h steps ahead.  shift(0) is a no-op → backward-compatible for horizon=1.
    if horizon > 1:
        result = (
            result
            .sort(_SLOT_COLS + ["t"])
            .with_columns(
                pl.col(col_name)
                  .shift(horizon - 1)
                  .over(_SLOT_COLS)
                  .alias(col_name)
            )
        )

    return result


# ===========================================================================
# B3 — Simple Exponential Smoothing (α=0.3, per slot, online)
# ===========================================================================

def _ses_one_slot(slot_df: pl.DataFrame, alpha: float) -> pl.DataFrame:
    """Online SES recursion for a single slot.

    s_t = α·y_t + (1-α)·s_{t-1}  (null observations skip the update).
    pred[i] = s before observing y[i]  (causal: shift-1 semantics).

    Initialization: s = NaN until the first non-null y_t; the first non-null
    value sets s directly (no prior needed) — AC-B3-3.  The prediction at
    that initialization row is NaN (no prior state), so the first test
    prediction for a slot with at least one train observation is the state
    after consuming ALL train rows.
    """
    slot_df = slot_df.sort("t")
    y = slot_df["delta_t_min"].to_numpy(allow_copy=True).astype(np.float64)
    pred = np.full(len(y), np.nan)
    s = np.nan  # smoothing state; NaN until first non-null

    for i in range(len(y)):
        # Prediction at row i is the state BEFORE observing y[i].
        pred[i] = s
        # Update state if current observation is not null/NaN.
        if not np.isnan(y[i]):
            if np.isnan(s):
                s = y[i]  # initialization: first non-null sets state directly
            else:
                s = alpha * y[i] + (1.0 - alpha) * s

    # Convert float NaN → polars null so downstream is_null() works correctly.
    pred_series = pl.Series("y_pred_b3", pred, dtype=pl.Float64)
    return slot_df.with_columns(pred_series.set(pred_series.is_nan(), None))


def predict_b3(headways: pl.DataFrame, *, alpha: float = SES_ALPHA, horizon: int = 1) -> pl.DataFrame:
    """Add column `y_pred_b3`: online SES predictions, α=0.3 (default).

    Applies the causal recursion s_t = α·y_t + (1-α)·s_{t-1} per slot.
    Null observations do not update the state (AC-B3-2).
    State is initialized from the first non-null observation (AC-B3-3).
    Slots with all-null values emit null for all rows (AC-B3-4).

    Horizon rule (Fase 6.5):
      The 1-step SES predictions are computed first (existing per-slot loop).
      Then shift(horizon-1) is applied over the slot key so that ŷ_{t+h} =
      ŷ_{t+1} lagged by h-1 additional steps.
      horizon=1 → shift(0) = identity (backward-compatible).

    Parameters
    ----------
    headways:
        headways DataFrame with `split` column attached.
    alpha:
        Smoothing parameter.  Default SES_ALPHA = 0.3 (locked, design §3).
        Tests may pass alternative values for edge-case verification.
    horizon:
        Prediction horizon in steps.  Default 1 reproduces the original
        behavior exactly.

    Returns
    -------
    pl.DataFrame — input frame with `y_pred_b3` (Float64 nullable) added.

    Design note (D-PL-OVER-VS-MAPGROUPS):
        polars .over() does not support stateful numpy loops; map_groups
        materializes one Python frame per slot (~30–100 per corridor — trivially fast).
    """
    sorted_df = headways.sort(_SLOT_COLS + ["t"])
    slots = sorted_df.partition_by(_SLOT_COLS, maintain_order=True)
    result = pl.concat([_ses_one_slot(g, alpha) for g in slots])

    # Apply horizon shift: shift(horizon-1) over slot so predictions look
    # h steps ahead.  shift(0) is a no-op → backward-compatible for horizon=1.
    if horizon > 1:
        result = (
            result
            .sort(_SLOT_COLS + ["t"])
            .with_columns(
                pl.col("y_pred_b3")
                  .shift(horizon - 1)
                  .over(_SLOT_COLS)
                  .alias("y_pred_b3")
            )
        )

    return result


# ===========================================================================
# B4 — Historical Average per (slot, hour-of-day) from train only
# ===========================================================================

def predict_b4_ha(headways: pl.DataFrame) -> pl.DataFrame:
    """Add column `y_pred_b4_ha`: per-slot, per-hour mean of train delta_t_min.

    For each (empresaid, direction, pair_rank, hour) group, computes the mean
    of non-null delta_t_min values from train rows only.  Test/val rows at the
    same hour receive that mean as their prediction.  Hours not seen in train
    produce null predictions.
    """
    ha_key = _SLOT_COLS + ["_hour"]
    df = headways.with_columns(pl.col("t").dt.hour().alias("_hour"))

    train_means = (
        df
        .filter(pl.col("split") == "train")
        .group_by(ha_key)
        .agg(pl.col("delta_t_min").mean().alias("y_pred_b4_ha"))
    )

    result = df.join(train_means, on=ha_key, how="left")
    return result.drop("_hour")

## Module: data/context_features

`load_atypical_days` + `encode_context` — the SAME helpers the DL notebooks use
to build the atypical-day flag. B5_XGB consumes `atypical_flag` through them, so
the fitted baseline and the DL models cannot diverge on what an atypical day is.
Must be embedded BEFORE fitted, which imports `encode_context`.

In [ ]:
"""Context features module for supervised dataset construction — Fase 3 DL.

AC-CTX-1: encode_context adds hour_sin, hour_cos at midnight → (0, 1).
AC-CTX-2: encode_context adds dow_sin, dow_cos with period 7; emits 5 named columns.
AC-CTX-3: load_atypical_days(None) returns empty set (graceful fallback, DL-2).
AC-CTX-4: load_atypical_days(path) returns set[date] from CSV when file exists.
AC-CTX-5: atypical_flag=1.0 when timestamp date in atypical_dates, else 0.0.
AC-CTX-6: zero torch imports at module level (INV-10, DL-10).

Design decisions locked in design §2.4 and §5:
  - encode_context operates on a DataFrame with a `t` (Datetime) column.
  - Cyclical encoding: sin(2π * value / period), cos(2π * value / period).
  - atypical_flag = 1.0 when t.date() in atypical_dates else 0.0.
  - DL-2: graceful fallback to atypical_flag=0 when path is None or missing.
  - No torch imports (INV-10).
"""
from __future__ import annotations

import logging
import math
import warnings
from datetime import date
from pathlib import Path

import polars as pl

_log = logging.getLogger(__name__)

# ---------------------------------------------------------------------------
# Constants
# ---------------------------------------------------------------------------

CONTEXT_FEATURE_NAMES: tuple[str, ...] = (
    "hour_sin",
    "hour_cos",
    "dow_sin",
    "dow_cos",
    "atypical_flag",
)


# ---------------------------------------------------------------------------
# Internal helpers
# ---------------------------------------------------------------------------

def _cyclical_pair(col: pl.Expr, period: int, prefix: str) -> list[pl.Expr]:
    """Emit [sin_expr, cos_expr] aliased <prefix>_sin, <prefix>_cos.

    Encoding: sin(2π * col / period), cos(2π * col / period).

    Parameters
    ----------
    col:
        Polars expression that yields a numeric value (e.g. hour 0-23, dow 0-6).
    period:
        Full cycle length (24 for hour, 7 for day-of-week).
    prefix:
        Column name prefix ("hour" or "dow").
    """
    angle = col * (2.0 * math.pi / period)
    return [
        angle.sin().alias(f"{prefix}_sin"),
        angle.cos().alias(f"{prefix}_cos"),
    ]


# ---------------------------------------------------------------------------
# Public API
# ---------------------------------------------------------------------------

def encode_context(
    df: pl.DataFrame,
    *,
    atypical_dates: set[date] | None = None,
) -> pl.DataFrame:
    """Add 5 context columns derived from the `t` (Datetime) column.

    AC-CTX-1..5. DL-2 graceful fallback: atypical_flag=0 when atypical_dates
    is None or empty.

    Parameters
    ----------
    df:
        DataFrame with a `t` (Datetime[us]) column.
    atypical_dates:
        Set of dates that are atypical (e.g. holidays, strikes). When None
        or empty, atypical_flag is 0.0 for all rows.

    Returns
    -------
    pl.DataFrame — input frame with 5 additional columns appended in the order
    defined by CONTEXT_FEATURE_NAMES.
    """
    if atypical_dates is None:
        atypical_dates = set()

    # Cyclical hour and day-of-week encodings.
    # polars dt.weekday() returns ISO weekday: Monday=1 .. Sunday=7.
    # We convert to 0-indexed (Monday=0 .. Sunday=6) to align with Python convention
    # so that midnight Monday → dow=0 → dow_sin=sin(0)=0, dow_cos=cos(0)=1 (AC-CTX-1).
    hour_expr = pl.col("t").dt.hour().cast(pl.Float64)
    dow_expr = (pl.col("t").dt.weekday() - 1).cast(pl.Float64)

    sin_cos_exprs: list[pl.Expr] = [
        *_cyclical_pair(hour_expr, 24, "hour"),
        *_cyclical_pair(dow_expr, 7, "dow"),
    ]

    # Atypical flag: 1.0 if the date is in the atypical set, else 0.0.
    if atypical_dates:
        # Build a list of date literals to check membership against.
        atypical_list = sorted(atypical_dates)
        date_col = pl.col("t").dt.date()
        flag_expr = pl.lit(0.0)

        # Chain when/then for each atypical date.
        flag_chain = pl.when(
            date_col == pl.lit(atypical_list[0])
        ).then(pl.lit(1.0))
        for d in atypical_list[1:]:
            flag_chain = flag_chain.when(
                date_col == pl.lit(d)
            ).then(pl.lit(1.0))
        flag_expr = flag_chain.otherwise(pl.lit(0.0))
    else:
        flag_expr = pl.lit(0.0)

    return df.with_columns(
        *sin_cos_exprs,
        flag_expr.cast(pl.Float64).alias("atypical_flag"),
    )


def load_atypical_days(
    path: Path | str | None,
) -> set[date]:
    """Read CSV with at least a `date` column; return set[date].

    AC-CTX-3 + DL-2: returns empty set when path is None OR file does not exist.
    A warning is emitted when the path is non-None but missing (so callers know
    the fallback was triggered — not a silent failure).

    Parameters
    ----------
    path:
        Path to a CSV file with a `date` column (ISO-8601 format).
        May be None, a string, or a Path object.

    Returns
    -------
    set[date] — parsed dates, or empty set on fallback.
    """
    if path is None:
        return set()

    resolved = Path(path)
    if not resolved.exists():
        warnings.warn(
            f"load_atypical_days: file not found at '{resolved}'; "
            "falling back to empty atypical set (atypical_flag=0 for all rows). "
            "DL-2 graceful fallback.",
            stacklevel=2,
        )
        return set()

    df = pl.read_csv(resolved, try_parse_dates=True)
    # The frozen 02-eda-corridors CSV names its date column `day`; older
    # fixtures use `date`. Accept either, preferring `date` when both exist.
    date_col = next((c for c in ("date", "day") if c in df.columns), None)
    if date_col is None:
        warnings.warn(
            f"load_atypical_days: CSV at '{resolved}' has no 'date' or 'day' column; "
            "falling back to empty set.",
            stacklevel=2,
        )
        return set()

    dates: set[date] = set()
    for val in df[date_col].to_list():
        if val is not None:
            if isinstance(val, date):
                dates.add(val)
            else:
                try:
                    from datetime import datetime as _dt
                    dates.add(_dt.fromisoformat(str(val)).date())
                except ValueError:
                    _log.warning("Skipping unparseable date value: %s", val)

    return dates

## Module: baselines/fitted

`predict_b5_xgb` / `fit_predict_b5_xgb` — fitted gradient-boosted baseline
(B5_XGB). Sees the same 12-lag window as the DL models (lag_1 == B1 persistence)
plus calendar/slot features AND the atypical-day flag. Runs a seeded
24-configuration random search selected strictly on the VALIDATION split.
Must be embedded BEFORE harness, which calls it.

In [ ]:
"""Fitted ML baseline for headway forecasting — gradient-boosted regressor (B5_XGB).

Why this module is separate from `statistical.py`:
    B0-B4 are closed-form/recursive predictors with NO learned parameters and a
    "no new dependencies" design lock. B5_XGB is a *fitted* learner (XGBoost) —
    a different category. It answers the reviewer reflex "where is a fitted/ML
    baseline?" that pure naive baselines (persistence, moving average, SES,
    historical average) do not.

Design — fair comparison to the DL models (NB11-13, NB17-19):
    The DL models consume an input window of T_in = 12 consecutive 1-minute
    steps and predict the headway HORIZON steps after the last input step. The
    XGBoost baseline is given the SAME information: 12 lagged headway values
    ending HORIZON steps before the target, so `lag_1` equals the B1 persistence
    prediction (`shift(horizon)`) and the model strictly extends the naive
    baselines rather than seeing extra future data. Calendar context (hour,
    weekday), static slot keys (direction, pair_rank) and the atypical-day flag
    round out the features.

    Two asymmetries versus the DL models were removed (peer-review fix):
      1. ATYPICAL-DAY FLAG. The DL models receive `atypical_flag` as a required,
         hash-pinned context feature. B5_XGB now receives the same binary flag,
         built with the SAME `encode_context` helper so the semantics cannot
         drift between the two model families.
      2. HYPERPARAMETER SEARCH. The DL models were tuned; B5_XGB used a single
         hardcoded configuration. It now runs a seeded random search of
         `SEARCH_N_CONFIGS` configurations selected STRICTLY on the validation
         split (see `_random_search`), with the winning configuration reported
         back to the caller so it can be persisted and audited.

Contract (mirrors statistical.py):
    predict_b5_xgb(headways, *, horizon=1, seed=42, atypical_dates=None,
                   search=True) -> headways + y_pred_b5_xgb
    fit_predict_b5_xgb(...) -> B5FitResult (predictions + search provenance)

    Input must have the `split` column (added by split_temporal). The model is
    fit on TRAIN rows only; predictions are produced for ALL rows. Validation
    rows are used for hyperparameter selection and early stopping ONLY when
    there are enough of them (>= _MIN_VAL_ROWS); otherwise the frozen default
    configuration and a fixed number of trees are used.

Leakage contract (hard):
    The `test` split NEVER influences training, early stopping, or
    hyperparameter selection. Selection reads the validation loss only.

Atypical-flag contract (mirrors the DL notebooks):
    `atypical_dates=None` means "no atypical calendar supplied" (library/fixture
    use) and yields an all-zero flag column. Passing an EXPLICIT EMPTY SET is a
    configuration error — a CSV that parsed to nothing must fail closed instead
    of silently disabling the feature — and raises ValueError.

Determinism:
    Fixed `seed`, fixed `SEARCH_SEED` for the configuration sampler, and
    `tree_method="hist"` with a pinned `nthread` → repeated calls on the same
    machine with the same inputs produce identical predictions and select the
    same configuration.
"""
from __future__ import annotations

from dataclasses import dataclass, field
from datetime import date

import numpy as np
import polars as pl


_SLOT_COLS: list[str] = ["empresaid", "direction", "pair_rank"]

# Number of lagged headway steps fed to the model = DL input window (T_in).
N_LAGS: int = 12

# Use validation rows for search + early stopping only when there are at least
# this many; tiny test fixtures (and corridors with no val rows) fall back to
# the frozen default configuration and a fixed number of trees.
_MIN_VAL_ROWS: int = 50

# Threads for the Kaggle CPU kernel (4 vCPU). Pinned in source: XGBoost `hist`
# is reproducible for a FIXED thread count, so this value is part of the
# determinism contract and must not be made environment-dependent.
_NTHREAD: int = 4

_NUM_BOOST_ROUND: int = 400
_EARLY_STOPPING_ROUNDS: int = 30

# Cheaper budget for the search sweep; the winner is refit at the full budget.
_SEARCH_NUM_BOOST_ROUND: int = 200
_SEARCH_EARLY_STOPPING_ROUNDS: int = 20

# Frozen fallback configuration (the pre-search hardcoded baseline). Used when
# there is no usable validation split, or when `search=False`.
_XGB_PARAMS: dict = {
    "eta": 0.05,
    "max_depth": 6,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "min_child_weight": 5,
    "lambda": 1.0,
    "objective": "reg:squarederror",
    "tree_method": "hist",
    "nthread": _NTHREAD,
}

# ---------------------------------------------------------------------------
# Hyperparameter random search — validation-only selection.
# ---------------------------------------------------------------------------

# EXACTLY 24 configurations: the agreed budget for a Kaggle CPU kernel that
# must fit 2 corridors x 4 horizons within the session runtime limit.
SEARCH_N_CONFIGS: int = 24

# Fixed in source so the search is reproducible and cannot be silently
# re-rolled between runs. Changing this value changes the paper's numbers.
SEARCH_SEED: int = 20240718

# Discrete search space (|space| = 6*6*5*5*5*5 = 22500 >> 24 draws).
SEARCH_SPACE: dict[str, list] = {
    "eta": [0.02, 0.03, 0.05, 0.08, 0.12, 0.20],
    "max_depth": [3, 4, 5, 6, 8, 10],
    "min_child_weight": [1, 3, 5, 10, 20],
    "subsample": [0.6, 0.7, 0.8, 0.9, 1.0],
    "colsample_bytree": [0.6, 0.7, 0.8, 0.9, 1.0],
    "lambda": [0.5, 1.0, 2.0, 5.0, 10.0],
}


def sample_search_configs(
    n_configs: int = SEARCH_N_CONFIGS, *, seed: int = SEARCH_SEED
) -> list[dict]:
    """Draw `n_configs` DISTINCT hyperparameter configurations, deterministically.

    Sampling depends only on `seed` and `SEARCH_SPACE` — never on the data — so
    every corridor and horizon evaluates the same candidate set and the sweep is
    exactly reproducible.

    Returns
    -------
    list[dict] — each dict holds only the searched keys (eta, max_depth,
    min_child_weight, subsample, colsample_bytree, lambda).
    """
    rng = np.random.default_rng(seed)
    keys = sorted(SEARCH_SPACE)  # sorted → draw order independent of dict order
    seen: set[tuple] = set()
    configs: list[dict] = []
    # Bounded loop: the space is ~22500 wide, so 24 distinct draws are reached
    # almost immediately; the cap only guards against a shrunken space.
    for _ in range(n_configs * 1000):
        if len(configs) == n_configs:
            break
        values = tuple(
            SEARCH_SPACE[k][int(rng.integers(len(SEARCH_SPACE[k])))] for k in keys
        )
        if values in seen:
            continue
        seen.add(values)
        configs.append(dict(zip(keys, values)))
    if len(configs) != n_configs:
        raise ValueError(
            f"sample_search_configs: could only draw {len(configs)} distinct "
            f"configurations out of {n_configs} requested"
        )
    return configs


@dataclass(frozen=True)
class B5FitResult:
    """Predictions plus the provenance needed to audit the fitted baseline.

    Attributes
    ----------
    predictions:
        Input frame (sorted by slot, t) with `y_pred_b5_xgb` added.
    best_params:
        The full parameter dict handed to XGBoost for the final fit.
    best_val_rmse:
        Validation RMSE of the selected configuration (``nan`` when no search
        ran, i.e. no usable validation split).
    best_iteration:
        Boosting iteration chosen by early stopping (-1 when unavailable).
    n_configs_evaluated:
        How many configurations the search actually fit (0 when it was skipped).
    search_seed:
        Seed used to draw the candidate configurations.
    used_atypical_flag:
        True when a non-empty atypical calendar was supplied.
    """

    predictions: pl.DataFrame
    best_params: dict = field(default_factory=dict)
    best_val_rmse: float = float("nan")
    best_iteration: int = -1
    n_configs_evaluated: int = 0
    search_seed: int = SEARCH_SEED
    used_atypical_flag: bool = False


def _build_features(
    headways: pl.DataFrame,
    *,
    horizon: int,
    atypical_dates: set[date] | None = None,
) -> tuple[pl.DataFrame, list[str]]:
    """Return the frame sorted by (slot, t) with lag + context feature columns
    added, plus the list of feature column names.

    lag_k (k = 1..N_LAGS) = headway value (forward-filled within slot) observed
    `horizon + k - 1` steps before the target row. lag_1 == B1 persistence.

    `_atypical` is the SAME binary flag the DL models consume: it is produced by
    `encode_context`, not reimplemented here, so the two model families cannot
    diverge on what counts as an atypical day.

    Raises
    ------
    ValueError
        If `atypical_dates` is an explicit empty set (fail closed — see the
        module docstring's atypical-flag contract).
    """
    if atypical_dates is not None and len(atypical_dates) == 0:
        raise ValueError(
            "_build_features: atypical_dates parsed to an EMPTY set. The "
            "atypical-day feature must not be silently disabled — pass None "
            "only when no atypical calendar exists at all."
        )

    lag_exprs = [
        pl.col("delta_t_min")
        .forward_fill()
        .shift(horizon + k - 1)
        .over(_SLOT_COLS)
        .alias(f"_lag_{k}")
        for k in range(1, N_LAGS + 1)
    ]
    df = (
        encode_context(headways, atypical_dates=atypical_dates)
        .sort(_SLOT_COLS + ["t"])
        .with_columns(
            *lag_exprs,
            pl.col("t").dt.hour().alias("_hour"),
            pl.col("t").dt.weekday().alias("_weekday"),
            pl.col("atypical_flag").alias("_atypical"),
        )
    )
    feature_cols = (
        [f"_lag_{k}" for k in range(1, N_LAGS + 1)]
        + ["_hour", "_weekday", "direction", "pair_rank", "_atypical"]
    )
    return df, feature_cols


def _random_search(
    xgb,
    dtrain,
    dval,
    *,
    seed: int,
    n_configs: int,
    search_seed: int,
) -> tuple[dict, float, int, int]:
    """Fit `n_configs` candidates and keep the one with the lowest VALIDATION RMSE.

    The only signal read here is `booster.best_score` on `dval` — the test split
    is not part of either DMatrix, so selection cannot see it.

    Returns
    -------
    (best_params, best_val_rmse, best_iteration, n_evaluated)
    """
    best_params: dict = {}
    best_score = float("inf")
    best_iteration = -1
    n_evaluated = 0

    for candidate in sample_search_configs(n_configs, seed=search_seed):
        params = dict(_XGB_PARAMS, **candidate, seed=seed)
        booster = xgb.train(
            params,
            dtrain,
            num_boost_round=_SEARCH_NUM_BOOST_ROUND,
            evals=[(dval, "val")],
            early_stopping_rounds=_SEARCH_EARLY_STOPPING_ROUNDS,
            verbose_eval=False,
        )
        n_evaluated += 1
        score = float(booster.best_score)
        # Strict `<` → first-drawn config wins ties, keeping selection
        # deterministic for a fixed candidate order.
        if score < best_score:
            best_score = score
            best_params = params
            best_iteration = int(getattr(booster, "best_iteration", -1))

    return best_params, best_score, best_iteration, n_evaluated


def fit_predict_b5_xgb(
    headways: pl.DataFrame,
    *,
    horizon: int = 1,
    seed: int = 42,
    atypical_dates: set[date] | None = None,
    search: bool = True,
    n_configs: int = SEARCH_N_CONFIGS,
    search_seed: int = SEARCH_SEED,
) -> B5FitResult:
    """Fit B5_XGB and return predictions plus the auditable search provenance.

    Parameters
    ----------
    headways:
        headways DataFrame with the `split` column attached. Columns consumed:
        empresaid, t, direction, pair_rank, delta_t_min, split.
    horizon:
        Forecast horizon in steps. lag_1 = shift(horizon) so the 1-lag feature
        equals B1 persistence; horizon=1 is the default.
    seed:
        XGBoost training seed (reproducible tree construction).
    atypical_dates:
        Atypical-day calendar (the same set the DL models receive). None means
        "not supplied" → all-zero flag; an explicit empty set raises.
    search:
        When True (default) run the seeded validation-only random search. When
        False, or when there is no usable validation split, the frozen default
        configuration is used.
    n_configs / search_seed:
        Search budget and sampler seed. Both default to the frozen constants;
        overriding them is a test/debug affordance, not a production path.

    Returns
    -------
    B5FitResult
    """
    import xgboost as xgb

    original_cols = headways.columns
    df, feature_cols = _build_features(
        headways, horizon=horizon, atypical_dates=atypical_dates
    )
    used_atypical = bool(atypical_dates)

    is_train = df["split"] == "train"
    is_val = df["split"] == "val"
    target_present = df["delta_t_min"].is_not_null()

    train_mask = (is_train & target_present).to_numpy()
    n_train = int(train_mask.sum())

    # Degenerate: nothing to fit on → null predictions (mirrors B0 on empty slots).
    if n_train == 0:
        return B5FitResult(
            predictions=df.select(original_cols).with_columns(
                pl.lit(None, dtype=pl.Float64).alias("y_pred_b5_xgb")
            ),
            used_atypical_flag=used_atypical,
        )

    X_all = df.select(feature_cols).to_numpy().astype(np.float64)
    y_all = df["delta_t_min"].to_numpy().astype(np.float64)

    dtrain = xgb.DMatrix(X_all[train_mask], label=y_all[train_mask], missing=np.nan)
    dall = xgb.DMatrix(X_all, missing=np.nan)

    val_mask = (is_val & target_present).to_numpy()
    has_val = int(val_mask.sum()) >= _MIN_VAL_ROWS

    params = dict(_XGB_PARAMS, seed=seed)
    best_val_rmse = float("nan")
    best_iteration = -1
    n_evaluated = 0

    if has_val:
        # NOTE: dval holds VALIDATION rows only. The test split is absent from
        # every DMatrix built here, so neither early stopping nor hyperparameter
        # selection can read it.
        dval = xgb.DMatrix(X_all[val_mask], label=y_all[val_mask], missing=np.nan)
        if search:
            params, best_val_rmse, _search_iter, n_evaluated = _random_search(
                xgb,
                dtrain,
                dval,
                seed=seed,
                n_configs=n_configs,
                search_seed=search_seed,
            )
        # Refit the selected configuration at the full boosting budget, keeping
        # the existing early-stopping-on-validation behaviour.
        booster = xgb.train(
            params,
            dtrain,
            num_boost_round=_NUM_BOOST_ROUND,
            evals=[(dval, "val")],
            early_stopping_rounds=_EARLY_STOPPING_ROUNDS,
            verbose_eval=False,
        )
        best_val_rmse = float(booster.best_score)
        best_iteration = int(getattr(booster, "best_iteration", -1))
    else:
        booster = xgb.train(params, dtrain, num_boost_round=_NUM_BOOST_ROUND)

    preds = booster.predict(dall).astype(np.float64)

    return B5FitResult(
        predictions=df.select(original_cols).with_columns(
            pl.Series("y_pred_b5_xgb", preds, dtype=pl.Float64)
        ),
        best_params=dict(params),
        best_val_rmse=best_val_rmse,
        best_iteration=best_iteration,
        n_configs_evaluated=n_evaluated,
        search_seed=search_seed,
        used_atypical_flag=used_atypical,
    )


def predict_b5_xgb(
    headways: pl.DataFrame,
    *,
    horizon: int = 1,
    seed: int = 42,
    atypical_dates: set[date] | None = None,
    search: bool = True,
) -> pl.DataFrame:
    """Add column `y_pred_b5_xgb`: gradient-boosted forecast of delta_t_min.

    Thin wrapper over :func:`fit_predict_b5_xgb` for callers that only need the
    predictions. See that function for the full parameter documentation.

    Returns
    -------
    pl.DataFrame — input frame (sorted by slot, t) with `y_pred_b5_xgb`
        (Float64 nullable) added. If the train split has no usable rows, the
        column is all-null.
    """
    return fit_predict_b5_xgb(
        headways,
        horizon=horizon,
        seed=seed,
        atypical_dates=atypical_dates,
        search=search,
    ).predictions

## Module: baselines/harness

`run_corridor` composes split → winsorize → all baselines (B0-B4 plus the
fitted B5_XGB when include_fitted=True, the default) → metrics per
(direction × baseline), returning a tidy 48-row long DataFrame plus the
per-sample XGB residuals and the fitted-baseline search provenance.
`evaluate_corridor` is the metrics-only wrapper. Accepts `horizon` to thread h
to B1/B2/B3/B5_XGB and `atypical_dates` to feed B5_XGB the atypical-day flag.

In [ ]:
"""Evaluation harness for classical baseline comparison — Fase 3.

Public API:
    run_corridor(headways, corridor_name, ...) -> CorridorRun
        metrics + per-sample XGB residuals + fitted-baseline provenance.
    evaluate_corridor(headways: pl.DataFrame, corridor_name: str) -> pl.DataFrame
        Metrics only (thin wrapper over run_corridor; unchanged contract).

The function composes the full pipeline for one corridor:
    split_temporal → winsorize_train_p99 → predict_b0/b1/b2(×3)/b3/b4_ha
    [→ predict_b5_xgb when include_fitted]
    → filter test rows → compute MAE + RMSE per (direction, baseline)
    → return tidy long-form DataFrame.

Output schema (design §6):
    corridor   Utf8
    direction  Utf8   — "-1", "+1", "aggregate"
    baseline   Utf8   — "B0", "B1", "B2_w5", "B2_w10", "B2_w15", "B3", "B4_HA"
                        [, "B5_XGB" when include_fitted]
    metric     Utf8   — "MAE", "RMSE"
    value      Float64 — minutes

Rows per corridor: 3 directions × N baselines × 2 metrics.
    include_fitted=True  (default): N = 8 → 48 rows per corridor.
    include_fitted=False (formulaic-only): N = 7 → 42 rows per corridor.

Design decisions (locked in design §6 and §9):
  - "aggregate" direction = MAE/RMSE over POOLED test rows (both directions
    concatenated), NOT mean of per-direction metrics.
  - val rows are NEVER consumed by the formulaic baselines B0-B4 (B3-VAL-UNUSED).
    The fitted baseline B5_XGB MAY use val rows for early stopping (only when
    there are enough), which is correct practice for a learned model and mirrors
    how the DL models were tuned.
  - B5_XGB (the fitted ML baseline) adds an xgboost dependency; it lives in
    fitted.py and is opt-out via include_fitted=False.
  - harness.py does NOT read parquets or write CSV (notebook does those).
"""
from __future__ import annotations

from dataclasses import dataclass, field
from datetime import date

import polars as pl


# Map from prediction column name → display name for the output DataFrame.
_BASELINE_MAP: list[tuple[str, str]] = [
    ("y_pred_b0", "B0"),
    ("y_pred_b1", "B1"),
    ("y_pred_b2_w5", "B2_w5"),
    ("y_pred_b2_w10", "B2_w10"),
    ("y_pred_b2_w15", "B2_w15"),
    ("y_pred_b3", "B3"),
    ("y_pred_b4_ha", "B4_HA"),
]

# The fitted ML baseline is appended only when include_fitted=True.
_FITTED_ENTRY: tuple[str, str] = ("y_pred_b5_xgb", "B5_XGB")

# Per-sample residual export for the paired significance tests (DM / Wilcoxon).
# `t` is the join key: without it the XGBoost residuals cannot be paired with
# any other model's per-sample errors.
XGB_RESIDUAL_COLUMNS: list[str] = [
    "corridor",
    "direction",
    "horizon",
    "t",
    "y_true",
    "y_pred_xgb",
    "y_pred_persist",
]


@dataclass(frozen=True)
class CorridorRun:
    """Everything one corridor x horizon run produces.

    Attributes
    ----------
    metrics:
        Tidy long-form MAE/RMSE table (the historical `evaluate_corridor` output).
    residuals:
        Per-sample TEST residuals for B5_XGB paired with B1 persistence, with the
        `t` join key. Empty frame when `include_fitted=False`.
    fit_result:
        Provenance of the fitted baseline (winning hyperparameters, validation
        RMSE, search budget). None when `include_fitted=False`.
    """

    metrics: pl.DataFrame
    residuals: pl.DataFrame = field(default_factory=lambda: pl.DataFrame([]))
    fit_result: B5FitResult | None = None


def _direction_label(direction_val: int) -> str:
    """Signed direction label ("-1" / "+1") shared with the DL residual exports."""
    return f"+{direction_val}" if direction_val > 0 else str(direction_val)


def _build_xgb_residuals(
    test_df: pl.DataFrame, corridor_name: str, horizon: int
) -> pl.DataFrame:
    """Per-sample paired TEST residuals: B5_XGB vs B1 persistence.

    Keeps only samples where the target AND both predictions are present — the
    paired set the significance tests require.
    """
    return (
        test_df.filter(
            pl.col("delta_t_min").is_not_null()
            & pl.col("y_pred_b5_xgb").is_not_null()
            & pl.col("y_pred_b1").is_not_null()
        )
        .with_columns(
            pl.lit(corridor_name).alias("corridor"),
            pl.col("direction")
            .map_elements(_direction_label, return_dtype=pl.Utf8)
            .alias("direction"),
            pl.lit(horizon, dtype=pl.Int64).alias("horizon"),
            pl.col("delta_t_min").cast(pl.Float64).alias("y_true"),
            pl.col("y_pred_b5_xgb").cast(pl.Float64).alias("y_pred_xgb"),
            pl.col("y_pred_b1").cast(pl.Float64).alias("y_pred_persist"),
        )
        .select(XGB_RESIDUAL_COLUMNS)
        .sort(["corridor", "direction", "t"])
    )


def run_corridor(
    headways: pl.DataFrame,
    corridor_name: str,
    *,
    horizon: int = 1,
    include_fitted: bool = True,
    atypical_dates: set[date] | None = None,
) -> CorridorRun:
    """Full pipeline for one corridor: metrics + XGB residuals + fit provenance.

    Same pipeline and contracts as :func:`evaluate_corridor` (which delegates
    here and returns only `metrics`), plus the two artifacts the paper needs to
    defend the fitted baseline: the per-sample paired residuals and the winning
    hyperparameter configuration.

    Parameters
    ----------
    atypical_dates:
        Atypical-day calendar forwarded to B5_XGB so the fitted baseline sees
        the same context feature as the DL models. An explicit empty set raises
        (fail closed); None means no calendar was supplied.
    """
    # --- Pipeline: split → winsorize → all baselines ---
    # INV-6: the p99 threshold is computed on TRAIN only and applied to ALL
    # splits — winsorize_train_p99 receives the full split-tagged frame.
    df = split_temporal(headways)
    df, _threshold = winsorize_train_p99(df)

    df = predict_b0(df)
    df = predict_b1(df, horizon=horizon)
    for w in BASELINE_B2_WINDOWS:
        df = predict_b2(df, window=w, horizon=horizon)
    df = predict_b3(df, horizon=horizon)
    df = predict_b4_ha(df)

    baseline_map = list(_BASELINE_MAP)
    fit_result: B5FitResult | None = None
    if include_fitted:
        fit_result = fit_predict_b5_xgb(
            df, horizon=horizon, atypical_dates=atypical_dates
        )
        df = fit_result.predictions
        baseline_map = baseline_map + [_FITTED_ENTRY]

    # --- Filter to test rows only (B3-VAL-UNUSED) ---
    test_df = df.filter(pl.col("split") == "test")

    metrics = _metrics_table(test_df, corridor_name, baseline_map)
    residuals = (
        _build_xgb_residuals(test_df, corridor_name, horizon)
        if include_fitted
        else pl.DataFrame([])
    )
    return CorridorRun(metrics=metrics, residuals=residuals, fit_result=fit_result)


def _metrics_table(
    test_df: pl.DataFrame,
    corridor_name: str,
    baseline_map: list[tuple[str, str]],
) -> pl.DataFrame:
    """MAE/RMSE per (direction x baseline) over the TEST rows, long-form."""
    rows: list[dict] = []

    for pred_col, baseline_name in baseline_map:
        for direction_val in (-1, 1, "aggregate"):
            if direction_val == "aggregate":
                # Pool all test rows regardless of direction.
                subset = test_df
                direction_str = "aggregate"
            else:
                subset = test_df.filter(pl.col("direction") == direction_val)
                direction_str = _direction_label(direction_val)

            y_true = subset["delta_t_min"]
            y_pred = subset[pred_col]

            # Compute MAE and RMSE (null rows are masked inside the functions).
            mae_val = mae(y_true, y_pred)
            rmse_val = rmse(y_true, y_pred)

            rows.append(
                {
                    "corridor": corridor_name,
                    "direction": direction_str,
                    "baseline": baseline_name,
                    "metric": "MAE",
                    "value": mae_val,
                }
            )
            rows.append(
                {
                    "corridor": corridor_name,
                    "direction": direction_str,
                    "baseline": baseline_name,
                    "metric": "RMSE",
                    "value": rmse_val,
                }
            )

    return pl.DataFrame(rows).with_columns(
        pl.col("corridor").cast(pl.Utf8),
        pl.col("direction").cast(pl.Utf8),
        pl.col("baseline").cast(pl.Utf8),
        pl.col("metric").cast(pl.Utf8),
        pl.col("value").cast(pl.Float64),
    )


def evaluate_corridor(
    headways: pl.DataFrame,
    corridor_name: str,
    *,
    horizon: int = 1,
    include_fitted: bool = True,
    atypical_dates: set[date] | None = None,
) -> pl.DataFrame:
    """Run all classical baselines on one corridor and return a tidy metrics table.

    Thin wrapper over :func:`run_corridor` kept for the existing callers that
    only need the metrics table.

    Parameters
    ----------
    headways:
        Raw headways DataFrame with R7 v4 schema columns:
        empresaid, t, direction, pair_rank, delta_t_min.
        Must NOT already have a `split` column (this function adds it).
    corridor_name:
        Label for the `corridor` column in the output (e.g. "E2", "E59").
    horizon:
        Prediction horizon in steps. Default 1 reproduces the original behavior
        exactly. B1, B2, B3, and B5_XGB are horizon-aware and receive this value.
        B0 and B4_HA are horizon-agnostic (constant/lookup predictors) and are
        not affected.
    include_fitted:
        When True (default), also runs the fitted ML baseline B5_XGB (xgboost).
        When False, only the formulaic baselines B0-B4 run (no xgboost import).

    Returns
    -------
    pl.DataFrame — tidy long-form table (48 rows with include_fitted, else 42):
        [corridor, direction, baseline, metric, value]

    Notes
    -----
    - val rows are ignored at prediction time (baselines consume train only)
      and are never included in metric computation (metrics use test rows only).
    - The "aggregate" direction row pools test rows from both directions before
      computing MAE/RMSE — it is NOT the mean of the two per-direction metrics.
    """
    return run_corridor(
        headways,
        corridor_name,
        horizon=horizon,
        include_fitted=include_fitted,
        atypical_dates=atypical_dates,
    ).metrics

## Cargar datos — E2 y E59

Lee los parquets generados por el notebook 04 (two-pass pipeline).

In [ ]:

# `empresaid` is implicit in the filename — inject it as a literal column
# so it matches the baselines contract (slot key requires it).
hw_e2  = pl.read_parquet(_find_parquet(2)).with_columns(pl.lit(2,  dtype=pl.Int64).alias("empresaid"))
hw_e59 = pl.read_parquet(_find_parquet(59)).with_columns(pl.lit(59, dtype=pl.Int64).alias("empresaid"))

print(f"E2:  {hw_e2.height:,} rows, {hw_e2.width} cols")
print(f"E59: {hw_e59.height:,} rows, {hw_e59.width} cols")

## Sanidad — cobertura por (empresa, dirección)

Fracción de `delta_t_min` no-nulo y conteo de filas por dirección.
Permite detectar baja cobertura en dir=+1 (riesgo R-DIR1-COVERAGE).

In [ ]:

for label, hw in [("E2", hw_e2), ("E59", hw_e59)]:
    print(f"\n=== {label} ===")
    summary = (
        hw.with_columns(pl.col("t").dt.date().alias("day"))
          .group_by(["empresaid", "direction"])
          .agg([
              pl.len().alias("n_rows"),
              (pl.col("delta_t_min").is_not_null().sum() / pl.len()).alias("non_null_frac"),
          ])
          .sort(["empresaid", "direction"])
    )
    print(summary)

## Días atípicos — input requerido y verificado por hash

`atypical_days.csv` (salida de NB02, kernel_source `alexhuaracha/02-eda-corridors`)
alimenta el flag `atypical_flag` que ahora también recibe B5_XGB. Es un input
**requerido**: si falta, si sus bytes no coinciden con el snapshot congelado, o
si el set parsea vacío, el kernel falla ANTES de ajustar nada — nunca se degrada
en silencio a un flag todo-ceros.

In [ ]:

atypical_path = _resolve_input("atypical_days.csv")
atypical_dates = load_atypical_days(atypical_path)
if not atypical_dates:
    raise ValueError(f"atypical_days.csv parsed to an empty date set: {atypical_path}")
print(f"Atypical days loaded: {len(atypical_dates)} dates (path={atypical_path})")

## Ejecutar harness — loop multi-horizonte

Llama a `run_corridor` para E2 y E59 a cada horizonte h ∈ {1, 3, 5, 10} y
concatena todos los resultados agregando la columna `horizon`.

`run_corridor` devuelve además (a) los residuos por muestra de B5_XGB emparejados
con la persistencia B1 sobre las MISMAS muestras de test (con la clave `t`), y
(b) la configuración ganadora del random search de 24 configuraciones, elegida
**solo** sobre el split de validación. Ambos se persisten a CSV para auditoría.

In [ ]:

HORIZONS = [1, 3, 5, 10]
frames = []
resid_frames = []
search_rows = []
for h in HORIZONS:
    for label, hw in [("E2", hw_e2), ("E59", hw_e59)]:
        run = run_corridor(hw, label, horizon=h, atypical_dates=atypical_dates)
        frames.append(run.metrics.with_columns(pl.lit(h, dtype=pl.Int64).alias("horizon")))
        resid_frames.append(run.residuals)
        fit = run.fit_result
        search_rows.append({
            "corridor": label,
            "horizon": h,
            "n_configs_evaluated": fit.n_configs_evaluated,
            "search_seed": fit.search_seed,
            "best_val_rmse": fit.best_val_rmse,
            "best_iteration": fit.best_iteration,
            "used_atypical_flag": fit.used_atypical_flag,
            **{f"param_{k}": str(v) for k, v in sorted(fit.best_params.items())},
        })
        print(f"{label} h={h}: best_val_rmse={fit.best_val_rmse:.5f} "
              f"({fit.n_configs_evaluated} configs) params={fit.best_params}")

results = pl.concat(frames)
residuals = pl.concat(resid_frames)
search_config = pl.DataFrame(search_rows)
print(f"Total rows: {results.height}  (expected 384 = 4 horizons x 2 corridors x 48)")
print(results.head(10))

## Escribir CSVs — métricas, residuos y configuración ganadora

1. `baselines_results_multih.csv` — tabla long-form
   (`corridor, direction, baseline, metric, value, horizon`).
2. `xgb_residuals_multih.csv` — residuos por muestra de B5_XGB emparejados con
   B1 (`corridor, direction, horizon, t, y_true, y_pred_xgb, y_pred_persist`),
   entrada de los tests de significancia.
3. `xgb_search_config_multih.csv` — hiperparámetros ganadores por
   (corredor, horizonte), la semilla del search y el RMSE de validación.

In [ ]:

results.write_csv(CSV_OUT)
print(f"CSV written to: {CSV_OUT}")
print(f"Rows: {results.height}  Columns: {results.columns}")

# Per-sample paired residuals (B5_XGB vs B1) for the DM / Wilcoxon tests.
residuals.write_csv(RESID_OUT)
print(f"Residuals written to: {RESID_OUT}  ({residuals.height:,} rows)")

# Winning hyperparameters per (corridor, horizon) — audit trail for the search.
search_config.write_csv(SEARCH_OUT)
print(f"Search config written to: {SEARCH_OUT}")
print(search_config)

## Tabla resumen (wide format)

Pivote ancho para lectura humana: filas = (corridor, horizon, direction, metric),
columnas = baseline.

In [ ]:

wide = results.pivot(
    on="baseline",
    index=["corridor", "horizon", "direction", "metric"],
    values="value",
)
print(wide.sort(["corridor", "horizon", "direction", "metric"]))